# Limpieza de Equipos 




In [1]:
import pandas as pd
import re



In [2]:
# Usuario que registra (Israel Cel)
ETL_USER_ID = 11

# Vendor genérico
VENDOR_ID = 1  # POR ASIGNAR

# AssetState
ASSET_STATE_ID = 2  # Asignado

# ProductType
PRODUCT_TYPE_COMPUTADORA = 1
PRODUCT_TYPE_SERVIDOR = 2

In [3]:
COMPANY_MAP = {
    'ISLA17_42_': 1,   # ISLA17
    'HSPRO_33_': 2,    # HSPRO
    'PTOARENAS': 9,    # PTOARENAS
}

In [4]:
SITE_MAP = {
    # ISLA17 (CompanyID=1)
    'BEACHPALACE': 2,
    'MOON GRAND': 8,
    'MOON SUNRISE': 10,
    
    # HSPRO (CompanyID=2)
    'BLUEBAY': 15,
    'H10 OCEAN': 16,
    'HYATT ZILARA': 18,
    
    # PTOARENAS (CompanyID=9)
    'ROYALTON RIV': 41,
    'SIRENIS': 46,
    # ... etc
}

In [5]:
def find_site_id(tienda_name, company_id):
    """Busca el SiteID basándose en el nombre de la tienda"""
    
    name_upper = tienda_name.upper()  # "bluebay tab caja" → "BLUEBAY TAB CAJA"
    
    # Buscar cada palabra clave en el nombre
    for keyword, site_id in SITE_MAP.items():
        if keyword.upper() in name_upper:
            return site_id   # ¡Encontrado! Retorna el ID
    
    return None  # No encontró ninguno

In [6]:
DEPART_MAP = {
    'IT': 1,
    'AMIANI': 2,
    'PIER27': 3,
    'MULTI': 4,
    'JOYERIA': 5,
    'JOY': 5,        # Abreviación de JOYERIA
    'TABAQUERIA': 8,
    'TAB': 8,        # Abreviación de TABAQUERIA
    'CAJA': 12,
    # ... etc
}

In [7]:
def is_servidor(tienda_name):
    """Determina si es SERVIDOR basándose en el nombre"""
    
    if not tienda_name:
        return False
    
    name_upper = tienda_name.upper()
    
    # Si contiene alguna de estas palabras, es servidor
    return 'SERVIDOR' in name_upper or 'SVR' in name_upper or 'SERVER' in name_upper

In [8]:
def extract_marca(modelo):
    """Extrae la marca del modelo"""
    
    if not modelo:
        return 'SIN IDENTIFICAR'
    
    modelo_upper = str(modelo).upper()
    
    if 'DELL' in modelo_upper or 'OPTIPLEX' in modelo_upper:
        return 'DELL'
    elif 'LENOVO' in modelo_upper or 'THINKCENTRE' in modelo_upper:
        return 'LENOVO'
    elif 'HP' in modelo_upper:
        return 'HP'
    else:
        return 'SIN IDENTIFICAR'

In [9]:
def generate_sql(records):
    sql_lines = []
    
    # Encabezado
    sql_lines.append("USE InventarioIT;")
    sql_lines.append("GO")
    sql_lines.append("")
    sql_lines.append("DECLARE @ETLUserID INT = 11;")
    sql_lines.append("DECLARE @VendorID INT = 1;")
    sql_lines.append("DECLARE @NewDetailID INT;")  # ← Variable para capturar el ID
    
    # Por cada registro:
    for rec in records:
        
        # INSERT 1: AssetDetail (datos técnicos)
        sql_lines.append(f"""
INSERT INTO AssetDetail (ProductManuf, SerialNum, Model, ...)
VALUES ('{rec['product_manuf']}', '{rec['serial_num']}', '{rec['model']}', ...);
""")
        
        # CAPTURAR EL ID QUE SE ACABA DE GENERAR
        sql_lines.append("SET @NewDetailID = SCOPE_IDENTITY();")
        
        # INSERT 2: Asset (usa el ID capturado)
        sql_lines.append(f"""
INSERT INTO Asset (Name, VendorID, ..., AssetDetailID)
VALUES ('{rec['name']}', @VendorID, ..., @NewDetailID);
""")

In [10]:
all_records = []  
from etl_assets import process_csv, generate_sql

# 1. Procesar cada CSV
csv_files = [
        (r'C:\Users\RTX\Documents\GitHub\PT\PROYECTO_TERMINAL_INVENTARIO_TI\datos\TIENDAS ACTUALIZACIONES - ISLA17(42).csv', 'ISLA17_42_'),
        (r'C:\Users\RTX\Documents\GitHub\PT\PROYECTO_TERMINAL_INVENTARIO_TI\datos\TIENDAS ACTUALIZACIONES - HSPRO(33).csv', 'HSPRO_33_'),
        (r'C:\Users\RTX\Documents\GitHub\PT\PROYECTO_TERMINAL_INVENTARIO_TI\datos\TIENDAS ACTUALIZACIONES - PTOARENAS.csv', 'PTOARENAS'),
    ]
    
for filepath, company_name in csv_files:
    records = process_csv(filepath, company_name)  # Lee y mapea
    all_records.extend(records)                     # Acumula
    
    # 2. Generar SQL
sql_output = generate_sql(all_records)
    
    # 3. Guardar archivo
with open('etl_assets_final.sql', 'w') as f:
    f.write(sql_output)